# RSNA Knee: what we tried, what moved the leaderboard, and what did not

**Team theartificialteam** (kadirov0, miraysudemutlu) · final private rank ≈ 1,730 / 6,425 · best public 0.958 / private 0.942

This is a post-competition summary, written for anyone who wants to learn from a mid-table run rather than a winning one.
It contains **no competition data** (no images, reports or labels), only our own submission scores and methods.

**TL;DR**
1. Fixing silent pipeline bugs took us from **0.500 to 0.956** public; switching to the best public recipe gave **0.958**.
2. None of our own models (Raptor-style CoAtNet / DINOv2 / ConvNeXt, trained on report-derived labels) helped the public blend.
   Every blend weight lowered both public and private scores.
3. Our label improvements looked large on the 58 expert-labelled studies (+0.021) but were almost invisible on the leaderboard (+0.001).
4. The winners' decisive step was **labels read from the images** (vision LLMs reading the MRI together with the report). We never got there.


## 1. Every submission, public and private

Scores are copied from our submission history. "base" = the public 0.958 blend recipe
(heliosli, built on nartaa's Raptor 384/224 checkpoints and the R6 ConvNeXt bundle).


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

subs = pd.DataFrame([
    ("first submission (old pipeline, silent fallback)", 0.500, 0.500),
    ("ours alone: Raptor-80, report labels v2",          0.928, 0.913),
    ("ours alone: Raptor-80, pseudo labels v3",          0.929, 0.915),
    ("ours alone: Raptor-80, pseudo labels v4",          0.926, 0.911),
    ("public 0.957 blend (older base)",                  0.956, 0.940),
    ("older base 0.80 + ours (44-slice) 0.20",           0.953, 0.938),
    ("older base 0.85 + ours (80-slice) 0.15",           0.955, 0.939),
    ("base 0.958",                                       0.958, 0.942),
    ("base 0.958 + 2.5D reader (heliosli 0.9586)",       0.958, 0.942),
    ("base + ours 10%",                                  0.956, 0.941),
    ("base + ours 20%",                                  0.955, 0.939),
    ("base + ours/DINOv2 20%",                           0.955, 0.939),
    ("base + ours 30%",                                  0.953, 0.937),
], columns=["submission", "public", "private"])
pd.set_option("display.width", 140)
display(subs)


In [ ]:
w = pd.DataFrame({"weight of our model": [0, 0.10, 0.20, 0.30],
                  "public": [0.958, 0.956, 0.955, 0.953],
                  "private": [0.942, 0.941, 0.939, 0.937]})
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(w["weight of our model"], w["public"], "o-", label="public")
ax.plot(w["weight of our model"], w["private"], "s--", label="private")
ax.set_xlabel("rank-blend weight of our model (rest: public 0.958 base)")
ax.set_ylabel("macro ROC AUC")
ax.set_title("Adding a weaker, correlated model only hurts")
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()


Public and private moved together for every submission, so the public board was not misleading here;
our models were simply too weak (≈0.93 public alone) next to a 0.958 base built from the same model family.

## 2. What we built

| Part | What | Outcome |
|---|---|---|
| DICOM preprocessing | geometric slice order (ImagePositionPatient · normal), knee side from tags or image centre, fixed-mm field of view | file-name order correlated only ρ ≈ 0.1–0.3 with true order; side agreement 98.5% |
| Silent-failure guards | strict checkpoint loading, failure-rate limits, constant-prediction checks | our first submission had scored 0.500 because inference silently wrote priors |
| Report labels | mean of public LLM label sets (stevenleehans, Claude Opus 4.5 labels by NTejas-1, JEV HYBRID) | 0.893 → 0.901 on the 58 gold studies |
| Pseudo labels | report labels mixed with out-of-fold predictions of our image models (v3, v4) | gold 0.901 → 0.922; leaderboard +0.001 / −0.003 |
| Models | Raptor recipe (CoAtNet-384, 2.5D windows, per-finding attention MIL) on our own 80-slice corpus; DINOv2 ViT-B; ConvNeXt | public 0.926–0.929 alone |
| Local LLM labelling | Qwen2.5-7B and Qwen3.5-27B on the reports | 0.80–0.88 on gold, no gain over public label sets |
| Diagnostic | ran the exact inference pipeline on the 58 gold studies | same AUC as during training → no train/test preprocessing mismatch |


## 3. Lessons

1. **The 58 gold studies did not predict the leaderboard for us.** Our pseudo labels contained our own models' predictions,
   so measuring them on gold was optimistic. Several top teams split gold into a tuning part and an untouched check part;
   the 2nd-place team quotes about ±0.02 of chance movement on the 58.
2. **A blend member must be strong *and* different.** With the public models' error correlation around ρ ≈ 0.86, a new member needs roughly
   ≥ 0.949 public on its own to add +0.002 (simple binormal model; it reproduced our four blend submissions within ±0.0002, although
   strength and correlation could not be separated from four points). Our ≈0.93 models were neither strong enough nor different enough.
3. **Labels were the lever, and the strongest labels came from the images.** Reports miss mild findings that the expert graders marked.
   The winners had vision LLMs read the MRI together with the report (gold agreement ≈ 0.94–0.956 versus ≈ 0.90 from reports),
   and in some cases also produce region-of-interest boxes.
4. **Infrastructure costs real time.** Quota limits, CLI versions, dropped downloads, a queue started without its environment,
   an upload step that made folds retrain three times, and a watcher that missed a file cost us more GPU hours than any experiment.
5. **Measure your own model alone on the leaderboard early.** We learned its real strength (0.929) only three days before the end.


## 4. Credits

Our final submissions are built on public work: **heliosli** (0.958 / 0.9586 blend recipes, R6 bundle), **nartaa / Danial Zakaria**
(Raptor 384/224 checkpoints and the "0.949 with one CoAtNet" write-up), **dreaddevelopment** (Raptor training and inference code),
**goodpjw2008** (2.5D ConvNeXt MIL reader), label sets by **stevenleehans**, **pilkwang**, **NTejas-1** and **riadmohamed42**,
analyses by **leoprovorov** and **varshithakolupuri**, and the forum posts of **hengck23**. Thank you to RSNA, the annotating
radiologists and Kaggle, and congratulations to the winners for sharing such detailed write-ups.
